<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Exploratory Data Analysis with Statistical Rigour

*Session 4 · Notebook 05 · Practical · Coach version (no hypothesis testing)*

## About this notebook

This is the capstone for Session 4. You will run a complete exploratory data analysis (EDA) on a real **credit risk** dataset, bringing together the tools from the session: data quality checks (04_03), distribution and transformation checks (04_01), correlation analysis (04_04), and quantifying uncertainty with confidence intervals (04_01). The goal is not just to compute numbers, but to follow a disciplined workflow that ends in a **statistical profile report**: a short, evidence-based summary of what separates good credit risks from bad, with recommendations.

The work is organised as a sequence of parts. Each part has a short explanation and a set of exercises that build toward the final report.

## About the exercises

Every exercise has three cells: a **question** (markdown), an empty **`# Your turn`** cell for you to attempt it, and a **`# Coach answer`** cell with a worked solution. In a live session, try each one before revealing the answer. The coach version runs top to bottom, so later parts reuse variables created earlier.

## About the data

The **German Credit** dataset (UCI Statlog) describes 1,000 loan applicants, each labelled as a **good** or **bad** credit risk. It has 20 attributes: a mix of numeric features (`duration_months`, `credit_amount`, `age_years`, ...) and categorical features (`checking_status`, `credit_history`, `purpose`, `housing`, `employment_since`, ...). The coded values have been decoded to readable labels, and the target is the `risk` column (`good` / `bad`). The classes are imbalanced: 700 good, 300 bad, which is typical of credit data. The file lives at `../datasets/Session_4/german_credit.csv`.

**Data source:** [UCI Statlog (German Credit Data)](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data).

### Data dictionary

Each row is one loan applicant. The columns:

| Column | Description |
|---|---|
| `checking_status` | status of the existing checking account |
| `duration_months` | loan duration in months |
| `credit_history` | the applicant's past credit behaviour |
| `purpose` | reason for the loan (car, education, furniture, ...) |
| `credit_amount` | amount of credit requested |
| `savings_status` | balance in savings accounts or bonds |
| `employment_since` | how long the applicant has been employed |
| `installment_rate` | installment as a percentage of disposable income |
| `personal_status_sex` | marital status and sex |
| `other_debtors` | other debtors or guarantors on the loan |
| `residence_since` | years at the current residence |
| `property` | main property owned (real estate, savings, car, ...) |
| `age_years` | applicant age in years |
| `other_installment_plans` | other installment plans (bank, stores, none) |
| `housing` | housing situation (own, rent, for free) |
| `existing_credits` | number of existing credits at this bank |
| `job` | job type and skill level |
| `num_dependents` | number of dependents |
| `telephone` | whether a telephone is registered |
| `foreign_worker` | whether the applicant is a foreign worker |
| `risk` | **target**: credit risk, `good` or `bad` |

## Index

- [Part 0: Setup and first look](#p0)
- [Part 1: Data quality](#p1)
- [Part 2: Distributions and transformations](#p2)
- [Part 3: Correlation analysis](#p3)
- [Part 4: Confidence intervals for the bad rate](#p4)
- [Part 5: Segmented analysis](#p5)
- [Part 6: Capstone - the statistical profile report](#p6)
- [Challenge](#challenge)
- [Further practice](#further)

<a id="p0"></a>
# Part 0: Setup and first look

Load the libraries and the data, and get oriented: size, columns, types and the balance of the target.

**Documentation:** [pandas](https://pandas.pydata.org/docs/) - the core library for the exploratory analysis in this capstone (with `scipy.stats` for the statistical tests).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import mahalanobis
from scipy.stats import pearsonr, spearmanr
from IPython.display import display

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# credit = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/german_credit.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# credit = pd.read_csv(session_datasets_http["german_credit"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# credit = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/german_credit.csv", header=True, inferSchema=True).toPandas()
credit = pd.read_csv('../datasets/Session_4/german_credit.csv')
print('Shape:', credit.shape)
credit.head()

### Exercise 0.1: First look at the data

Get oriented before analysing. Print the **column types** with `credit.dtypes`, then look at the **target**: print the balance of the `risk` column with `value_counts()` and plot it as a bar chart. Is the dataset balanced?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
# Column types and the target balance
print(credit.dtypes)
print('\nRisk balance:')
print(credit['risk'].value_counts())
credit['risk'].value_counts().plot(kind='bar', color=['seagreen', 'indianred'])
plt.title('Target balance: good vs bad credit risk'); plt.ylabel('count'); plt.show()

### Exercise 0.2: Create the comparison groups

We will compare good vs bad risks throughout, so set up the pieces once: build a list of the **numeric** column names and a list of the **categorical** ones (excluding the target `risk`), and split the data into two DataFrames, `good` and `bad`, by the `risk` label.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
numeric_cols = credit.select_dtypes('number').columns.tolist()
categorical_cols = credit.select_dtypes('object').drop(columns='risk', errors='ignore').columns.tolist()
good = credit[credit['risk'] == 'good']
bad = credit[credit['risk'] == 'bad']
print('Numeric columns:', numeric_cols)
print('Categorical columns:', len(categorical_cols))

<a id="p1"></a>
# Part 1: Data quality

Before any analysis, check the data is sound (Session 04_03). Two things to do here:

- **Check completeness:** how many values are missing, and are there duplicate rows? Where values are missing, the mechanisms (MCAR/MAR/MNAR) and imputation methods from 04_03 would apply.
- **Detect outliers:** reuse the outlier functions from 04_03 (z-score, IQR and Mahalanobis distance), because a credit team must spot both single-variable extremes and applicants who are unusual in combination.

### Exercise 1.1: Bring across the outlier functions

We will reuse the outlier tools from 04_03. Bring these functions into this notebook so they work on the credit data: `detect_outliers_zscore`, `detect_outliers_iqr`, `detect_outliers_mahalanobis`, and the two handlers `remove_outliers` and `cap_outliers`. (Copy them across from 04_03; the coach answer shows them.)

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
def detect_outliers_zscore(df, features, threshold=3):
    """Row indices that are outliers on ANY of the features by the z-score rule."""
    idx = set()
    for f in features:
        z = (df[f] - df[f].mean()) / df[f].std()
        idx.update(df.index[np.abs(z) > threshold])
    return sorted(idx)

def detect_outliers_iqr(df, features):
    """Row indices that are outliers on ANY of the features by the IQR rule."""
    idx = set()
    for f in features:
        q1, q3 = df[f].quantile(0.25), df[f].quantile(0.75)
        iqr = q3 - q1
        idx.update(df.index[(df[f] < q1 - 1.5*iqr) | (df[f] > q3 + 1.5*iqr)])
    return sorted(idx)

def detect_outliers_mahalanobis(df, features, alpha=0.025):
    """Row indices whose Mahalanobis distance exceeds the chi-square cutoff."""
    X = df[features].to_numpy()
    centre = X.mean(axis=0)
    inv_cov = np.linalg.inv(np.cov(X.T))
    dist = np.array([mahalanobis(row, centre, inv_cov) for row in X])
    cutoff = np.sqrt(stats.chi2.ppf(1 - alpha, df=len(features)))
    return df.index[dist > cutoff].tolist(), dist, cutoff

def remove_outliers(df, indices):
    return df.drop(index=indices).reset_index(drop=True)

def cap_outliers(df, indices, features, method='median'):
    out = df.copy()
    clean_rows = out.drop(index=indices)
    for f in features:
        out.loc[indices, f] = getattr(clean_rows[f], method)()
    return out

print('Outlier functions ready:', [detect_outliers_zscore, detect_outliers_iqr,
      detect_outliers_mahalanobis, remove_outliers, cap_outliers].__len__(), 'functions')

### Exercise 1.2: Missing values and duplicates

Print the number of missing values per column and the number of duplicated rows. Is the data complete, or would you need the imputation techniques from 04_03?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
print('Missing values per column:')
print(credit.isna().sum()[credit.isna().sum() > 0] if credit.isna().any().any() else 'none')
print(f'\nDuplicated rows: {credit.duplicated().sum()}')
print('The dataset is complete, so no imputation is required.')

### Exercise 1.3: Univariate outliers (z-score vs IQR)

Use `detect_outliers_zscore` and `detect_outliers_iqr` on the three continuous features `['duration_months', 'credit_amount', 'age_years']`. How many rows does each method flag, and why might IQR flag more on this skewed data?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
feats = ['duration_months', 'credit_amount', 'age_years']
z_idx = detect_outliers_zscore(credit, feats)
iqr_idx = detect_outliers_iqr(credit, feats)
print(f'z-score flags {len(z_idx)} rows; IQR flags {len(iqr_idx)} rows')
print('These features are right-skewed, so IQR (quartile-based, robust) flags more of the '
      'long right tail than the z-score, whose mean and sd are themselves pulled by extremes.')

### Exercise 1.4: Multivariate outliers (Mahalanobis)

Use `detect_outliers_mahalanobis` on the same three features to find applicants who are unusual in combination (for example a young person with a very large, long loan). How many are flagged? Would you remove them?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
feats = ['duration_months', 'credit_amount', 'age_years']
idx, dist, cutoff = detect_outliers_mahalanobis(credit, feats)
print(f'Mahalanobis cutoff = {cutoff:.2f}; multivariate outliers = {len(idx)}')
print('These are unusual combinations worth a manual review, but they are plausible real '
      'applicants, so we keep them rather than remove them (remove_outliers/cap_outliers are '
      'available if a genuine error were found).')

<a id="p2"></a>
# Part 2: Distributions and transformations

Understand the shape of the key numeric variables (Session 04_01). We focus on the three main continuous features, `duration_months`, `credit_amount` and `age_years`; credit amounts and durations are usually right-skewed, which matters for the methods we choose later.

### Exercise 2.1: Plot the distributions

Plot the distribution of the three main numeric features (`duration_months`, `credit_amount`, `age_years`) as histograms with a KDE overlay, and note which ones look skewed.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
# Distributions of the main numeric features
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, col in zip(ax, ['duration_months', 'credit_amount', 'age_years']):
    sns.histplot(credit[col], kde=True, ax=a)
    a.set_title(f'{col} (skew = {stats.skew(credit[col]):.2f})')
plt.tight_layout(); plt.show()

### Exercise 2.2: Quantify the skew

Print the skewness of `duration_months`, `credit_amount` and `age_years`. Which is the most skewed?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
for col in ['duration_months', 'credit_amount', 'age_years']:
    print(f'{col:16}: skew = {stats.skew(credit[col]):.2f}')
print('credit_amount is the most right-skewed.')

### Exercise 2.3: Transform a skewed variable

Apply a log transform to `credit_amount`, store it as a new column `log_credit_amount`, and compare the skewness before and after. Plot a Q-Q plot of the transformed values.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
credit['log_credit_amount'] = np.log(credit['credit_amount'])
print(f"Skew before: {stats.skew(credit['credit_amount']):.2f}")
print(f"Skew after log: {stats.skew(credit['log_credit_amount']):.2f}")
stats.probplot(credit['log_credit_amount'], dist='norm', plot=plt)
plt.title('Q-Q plot of log(credit_amount)'); plt.show()

<a id="p3"></a>
# Part 3: Correlation analysis

Look at how the numeric features relate to each other (Session 04_04). Because some are skewed, we will glance at both Pearson and Spearman.

### Exercise 3.1: Correlation matrix

Build a Pearson correlation matrix of the original numeric columns (`numeric_cols`) and draw a masked lower-triangle heatmap. Which two features are most strongly related?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
corr = credit[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Numeric correlations (German Credit)'); plt.show()
print('duration_months and credit_amount are the most strongly correlated: longer loans are larger.')

### Exercise 3.2: Pearson vs Spearman on a skewed pair

For `credit_amount` vs `duration_months`, compute both Pearson and Spearman. Are they similar? What would a gap have told you?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
a, b = credit['credit_amount'], credit['duration_months']
print(f'Pearson  = {pearsonr(a, b)[0]:.3f}')
print(f'Spearman = {spearmanr(a, b)[0]:.3f}')
print('They are close, so the relationship is roughly monotonic and not dominated by outliers.')

<a id="p4"></a>
# Part 4: Quantifying uncertainty with confidence intervals

A number measured on a sample is an estimate, not a fact (Session 04_01). Instead of a single point value, we report a **confidence interval**: a plausible range for the true value. This is the honest way to present a KPI to management or a regulator, and it is the risk framing from 04_01 ("bad rate 30%, 95% CI 27% to 33%").

### A confidence interval for the bad rate

The bad rate is simply the **average of a 0/1 flag** (1 if the applicant is bad, 0 if good), so it is a *mean* and we can reuse the exact tool from 04_01: `stats.t.interval`. No new formula is needed. We build the flag once and read off the interval for the whole portfolio.

In [ ]:
credit['is_bad'] = (credit['risk'] == 'bad').astype(int)   # 1 = bad, 0 = good

mean_bad = credit['is_bad'].mean()
lo, hi = stats.t.interval(0.95, df=len(credit) - 1,
                          loc=mean_bad, scale=stats.sem(credit['is_bad']))
print(f'Overall bad rate = {mean_bad:.1%}, 95% CI [{lo:.1%}, {hi:.1%}]  (n = {len(credit)})')

### Confidence intervals per segment

The same one-liner, applied **within each** `checking_status` group, drawn as bars with error bars. A wide bar (a small group) means an uncertain estimate, which matters before acting on that segment.

In [ ]:
overall_bad = credit['is_bad'].mean()
rows = []
for grp, g in credit.groupby('checking_status'):
    m = g['is_bad'].mean()
    lo, hi = stats.t.interval(0.95, df=len(g) - 1, loc=m, scale=stats.sem(g['is_bad']))
    rows.append({'group': grp, 'bad_rate': m, 'lo': lo, 'hi': hi, 'n': len(g)})
seg = pd.DataFrame(rows).sort_values('bad_rate', ascending=False).reset_index(drop=True)
display(seg.round(3))

err = [seg['bad_rate'] - seg['lo'], seg['hi'] - seg['bad_rate']]
plt.figure(figsize=(8, 4))
plt.bar(seg['group'], seg['bad_rate'], yerr=err, capsize=5, color='indianred')
plt.axhline(overall_bad, color='black', ls='--', label='overall')
plt.ylabel('bad rate'); plt.title('Bad rate by checking status (95% CI)')
plt.xticks(rotation=15); plt.legend(); plt.tight_layout(); plt.show()

### A confidence interval for a mean

Exactly the same tool works for a numeric feature. Here is a 95% CI for the **mean credit amount** in each risk group. When the two intervals barely overlap it is a strong visual sign the groups genuinely differ, without any formal test.

In [ ]:
for label, g in [('good', good), ('bad', bad)]:
    m = g['credit_amount'].mean()
    lo, hi = stats.t.interval(0.95, df=len(g) - 1, loc=m, scale=stats.sem(g['credit_amount']))
    print(f'{label:4}: mean credit_amount = {m:,.0f}, 95% CI [{lo:,.0f}, {hi:,.0f}]')

### Exercise 4.1: Confidence intervals by housing

Using the same `stats.t.interval` approach on the `is_bad` flag, print the bad rate and its 95% CI for each `housing` group. Which group's interval is the widest, and why?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
for grp, g in credit.groupby('housing'):
    m = g['is_bad'].mean()
    lo, hi = stats.t.interval(0.95, df=len(g) - 1, loc=m, scale=stats.sem(g['is_bad']))
    print(f'{grp:9}: bad rate {m:.1%}, 95% CI [{lo:.1%}, {hi:.1%}]  (n = {len(g)})')
print('The smallest group has the widest interval: less data means a less precise estimate.')

<a id="p5"></a>
# Part 5: Segmented analysis

Confidence intervals tell us how precise an estimate is; **segmentation** shows us *where* the risk concentrates. For a categorical feature, we compute the **bad rate** (the share of applicants who are bad) within each of its categories and compare it against the overall rate. A category whose bad rate sits well above the portfolio average is a red flag a credit team acts on.

The pattern in code is `credit.groupby(<feature>)['is_bad'].mean()`: split the rows by the feature's categories, then average the 0/1 `is_bad` flag within each group, which gives that category's bad rate. The worked example below does this for `checking_status`.

In [ ]:
# Bad rate by a category, with the overall rate as a reference line
overall_bad = credit['is_bad'].mean()
print(f'Overall bad rate: {overall_bad:.1%}')

bad_rate_checking = credit.groupby('checking_status')['is_bad'].mean().sort_values(ascending=False)
bad_rate_checking.plot(kind='barh', color='indianred')
plt.axvline(overall_bad, color='black', ls='--', label='overall')
plt.title('Bad rate by checking account status'); plt.xlabel('bad rate'); plt.legend(); plt.show()
print((bad_rate_checking * 100).round(1))

### Exercise 5.1: Bad rate by credit history

Compute the bad rate within each `credit_history` category (group by `credit_history` and take the mean of `is_bad`), then sort from highest to lowest. Which credit-history category is the riskiest, and how does it compare with the overall bad rate?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
br = credit.groupby('credit_history')['is_bad'].mean().sort_values(ascending=False)
print((br * 100).round(1))
print('Applicants with no prior credits (or all paid elsewhere) show the highest bad rate.')

### Exercise 5.2: Bad rate by age band

Credit risk often changes with age. Create an age band with `pd.cut(credit['age_years'], bins=[18, 25, 35, 50, 75])`, compute the bad rate within each band (the mean of `is_bad`), and read off the trend. Are younger applicants riskier?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
credit['age_band'] = pd.cut(credit['age_years'], bins=[18, 25, 35, 50, 75])
br = credit.groupby('age_band', observed=True)['is_bad'].mean()
print((br * 100).round(1))
print('The youngest band (18-25) has the highest bad rate; risk falls with age.')

<a id="p6"></a>
# Part 6: Capstone - the statistical profile report

Pull everything together into a short, evidence-based **statistical profile** of credit risk in this portfolio. A good report does three things: ranks the features that most separate good from bad, states the findings in plain language, and makes recommendations. The three exercises below build it.

### Exercise 6.1: Rank the numeric features by association with risk

We want the numeric features that best separate good from bad risks.

1. First **plot the distribution** of each numeric feature (a histogram grid) and note which are skewed. This tells you which correlation coefficient is appropriate (recall 04_04: skewed or outlier-prone data favours the rank-based Spearman).
2. Then encode `risk` as a 0/1 column `is_bad` and **rank the features by the absolute correlation** with it, using the coefficient you chose. Save the table as `numeric_rank`. Which feature is most associated with being a bad risk?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
num_feats = ['duration_months', 'credit_amount', 'age_years', 'installment_rate',
             'residence_since', 'existing_credits', 'num_dependents']

# Step 1: distributions - several (credit_amount, duration_months) are right-skewed,
# so Spearman (rank-based, robust to skew and outliers) is the safer choice.
credit[num_feats].hist(figsize=(13, 7), bins=25, color='steelblue', edgecolor='white')
plt.suptitle('Distributions of the numeric features', y=1.02); plt.tight_layout(); plt.show()

# Step 2: rank by absolute Spearman correlation with the 0/1 target
credit['is_bad'] = (credit['risk'] == 'bad').astype(int)
numeric_rank = (credit[num_feats + ['is_bad']].corr(method='spearman')['is_bad']
                .drop('is_bad').rename('corr_with_is_bad').to_frame())
numeric_rank['abs_corr'] = numeric_rank['corr_with_is_bad'].abs()
numeric_rank = numeric_rank.sort_values('abs_corr', ascending=False)
display(numeric_rank.round(3))
print(f'Strongest numeric associate of risk: {numeric_rank.index[0]}')

### Exercise 6.2: Rank the categorical features by association with risk

For each categorical feature, compute **Cramer's V** with `risk` (the 0-to-1 association measure from 04_04), sort by it, and save the table as `cat_rank`. What are the top three drivers of risk?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
def cramers_v(x, y):
    """Association between two categorical variables, scaled 0 to 1 (from 04_04)."""
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.to_numpy().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

cat_rank = (pd.DataFrame({'feature': categorical_cols,
                          'cramers_v': [cramers_v(credit[c], credit['risk']) for c in categorical_cols]})
            .sort_values('cramers_v', ascending=False).reset_index(drop=True))
display(cat_rank.round(3))
print("Top three drivers by Cramer's V:", cat_rank.head(3)['feature'].tolist())

### Exercise 6.3: Write the statistical profile report

Using `numeric_rank` and `cat_rank`, write a short report (3 to 5 bullet points of findings, plus recommendations) summarising what drives credit risk in this portfolio. The coach answer shows one complete version.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
top_num = numeric_rank.index[0]
top_cats = cat_rank.head(3)['feature'].tolist()

print('=' * 70)
print('STATISTICAL PROFILE: drivers of credit risk (German Credit, n=1,000)')
print('=' * 70)
print(f'Overall bad rate: {overall_bad:.1%} (300 bad, 700 good); see Part 4 for its 95% CI.')
print()
print('Key findings:')
print(f'  1. The strongest categorical drivers of risk are: {", ".join(top_cats)} '
      "(highest Cramer's V association with the good/bad label).")
print(f'  2. Among numeric features, {top_num} is the most associated with risk '
      '(largest absolute Spearman correlation with the bad flag).')
print('  3. Bad rate is far above average for applicants with little or no checking balance, '
      'and for the youngest age band (18-25).')
print('  4. Loan size and duration move together and are both higher for bad risks, so large '
      'long loans deserve extra scrutiny.')
print()
print('Recommendations:')
print('  - Weight checking-account status and credit history heavily in the scorecard.')
print('  - Apply tighter limits or manual review to long-duration, high-amount applications.')
print('  - Treat the 18-25 segment as higher risk and price or review accordingly.')
print('  - Report each segment bad rate with its confidence interval, and remember the 70/30 '
      'class imbalance when building any model (Session 5).')

<a id="challenge"></a>
## Challenge (optional): which features correlate most with being a bad risk?

Bring the numeric and categorical drivers into a single view. One-hot encode the categorical features, add `risk` as a 0/1 column, build a correlation matrix, and read off which columns correlate most strongly (positively or negatively) with being a bad risk. This combines the encoding idea from 04_04 with the target-correlation idea from Part 6.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
enc = pd.get_dummies(credit[categorical_cols], drop_first=True).astype(int)
enc['is_bad'] = (credit['risk'] == 'bad').astype(int)
full = pd.concat([credit[numeric_cols], enc], axis=1)

target_corr = full.corr()['is_bad'].drop('is_bad').sort_values(key=abs, ascending=False)
print('Most associated with being a bad risk (top 10 by absolute correlation):')
print(target_corr.head(10).round(3))
print('\nMost protective (negative) associations:')
print(target_corr.tail(5).round(3))
print('\nThese one-hot columns line up with the Cramer\'s V ranking from Exercise 6.2.')

<a id="further"></a>

## Further practice

- Add a bad-rate table for `purpose` and `savings_status` with 95% confidence intervals, and flag any segment whose interval is very wide (a small sample).
- Create a single summary figure (small multiples) of bad rate across the top three categorical drivers.